In [ ]:
# Exploratory out-of-sample reversion check (B001)
# Formation-window β and intercept are frozen; every stat below uses the trading window only.
from AlgorithmImports import *
import numpy as np, pandas as pd
from statsmodels.tsa.stattools import adfuller
from pairs_core import repo_block

BATCH_ID = "B001"
F_START, F_END = datetime(2020, 12, 31), datetime(2023, 12, 31)   # formation (3y to AS_OF)
T_START, T_END = datetime(2024, 1, 1), datetime(2025, 12, 31)     # trading window (same as backtests)
MIN_F_DAYS, MIN_T_DAYS = 500, 120                                 # placeholder: minimum history to score a pair
GROUPS = {
    "skill":         "FBP/OFG,AVA/NWE,GPI/ABG,ONTO/CAMT,GNK/SBLK,GPRE/REX,ICHR/UCTT,WIRE/ATKR,UFPI/BCC,L/CNA,FMX/KOF,IEP/CVI",
    "random":        "G/OKTA,ERIC/JEF,ROK/QCOM,SPLK/CUZ,WPC/CMS,K/TFII,UEC/FRT,EDU/CWH,NVR/PEG,FDS/PRGO,CNK/KVUE,WST/ADI",
    "same_industry": "FFIV/MSFT,LNG/TRP,VRSN/ZS,GILD/OGN,FICO/UBER,AMT/IRM,OTIS/PH,NET/PANW,MU/TXN,ETN/GGG,AER/R,BPOP/CMA",
}

qb = QuantBook()

def log_close(t):
    """Adjusted daily log close, formation start to trading end."""
    h = qb.history(qb.add_equity(t).symbol, F_START, T_END, Resolution.DAILY)
    if h.empty:
        return None
    s = h["close"]
    if isinstance(s.index, pd.MultiIndex):
        s = s.droplevel(0)
    s.index = pd.to_datetime(s.index).normalize()
    return np.log(s[~s.index.duplicated()])

def half_life(s):
    """AR(1) half-life: -ln2 / lambda; inf if not mean-reverting."""
    lag = s.shift(1).dropna()
    ds = s.diff().dropna().loc[lag.index]
    lam = np.polyfit(lag.values, ds.values, 1)[0]
    return -np.log(2) / lam if lam < 0 else np.inf

def score_pair(a, b):
    la, lb = log_close(a), log_close(b)
    if la is None or lb is None:
        return dict(status="no data")
    df = pd.concat([la, lb], axis=1, keys=["a", "b"]).dropna()
    f, t = df.loc[F_START:F_END], df.loc[T_START:T_END]
    if len(f) < MIN_F_DAYS or len(t) < MIN_T_DAYS:
        return dict(status=f"short (f={len(f)}, t={len(t)})")
    beta, alpha = np.polyfit(f.b, f.a, 1)                  # frozen at the as-of date
    sf = f.a - beta * f.b - alpha
    st = t.a - beta * t.b - alpha                          # out-of-sample spread
    return dict(
        status="ok",
        beta_f=beta,
        beta_t=np.polyfit(t.b, t.a, 1)[0],                 # refit, for drift only
        adf_p_t=adfuller(st, autolag="AIC")[1],
        hl_f=half_life(sf),
        hl_t=half_life(st),
        cross_yr_t=(np.sign(st) * np.sign(st.shift(1)) < 0).sum() / (len(t) / 252),
        std_ratio=st.std() / sf.std(),                     # >>1 = spread blew out
        mean_shift=abs(st.mean()) / sf.std(),              # drift away from formation mean, in formation SDs
        t_days=len(t),
    )

rows = []
for g, txt in GROUPS.items():
    for p in txt.split(","):
        a, b = p.split("/")
        try:
            r = score_pair(a, b)
        except Exception as e:
            r = dict(status=f"error: {e}")
        rows.append(dict(group=g, pair=p, **r))
rv = pd.DataFrame(rows)

ok = rv[rv.status == "ok"].copy()
ok["reverts"] = (ok.adf_p_t < 0.05) & ok.hl_t.between(5, 60)
summary = ok.groupby("group", sort=False).agg(
    scored=("pair", "size"),
    reverts=("reverts", "sum"),
    adf_lt_05=("adf_p_t", lambda x: (x < 0.05).sum()),
    adf_p_med=("adf_p_t", "median"),
    hl_t_med=("hl_t", "median"),
    cross_yr_med=("cross_yr_t", "median"),
    std_ratio_med=("std_ratio", "median"),
)
print(summary.round(3).to_string())
print(rv[rv.status != "ok"][["group", "pair", "status"]].to_string())
print(repo_block("reversion.csv", rv.round(4).to_csv(index=False)))